# Table 3 and Fig. 6: the three generalised forms

Each form is fitted to the median of the 30 curves in Halo 2's frame. In $\log_{10} f$ all
three are linear in their parameters, so every fit is a least-squares solve:

1. $\log_{10} f = a + \alpha \log_{10}[\hat\varepsilon/(1-\hat\varepsilon)]$
2. $\log_{10} f = a + \alpha \log_{10}\hat\varepsilon - \beta \log_{10}(1-\hat\varepsilon)$
3. $\log_{10} f = a + \alpha \log_{10}[\hat\varepsilon/(1-\hat\varepsilon)] - \hat\varepsilon_0/(\hat\varepsilon \ln 10)$

with $C = 10^a G^{3/2} r_s^3 \rho_s^{1/2}$ from Halo 2's NFW fit. Errors come from a halo
bootstrap (resample the 30 halos, retake the median, refit), quoted as the larger side of
the 16-84 percentile interval.

Reads the CV test curves and `folds.json` (`$CV_DIR`) and Halo 2's density fit.
Writes `forms_params.csv`, `forms_per_halo.csv`, `table3.tex`,
`fig6_{a,b,c}.dat` and `fig6_meta.gnu` to `output/`. Figs. 7 and 8 read the first two.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports

import numpy as np
import pandas as pd

from Empirical_DF_Construction.paper_figures.common import G, REF_HALO, density_fit, load_cv, out_dir

N_BOOT = 100_000
SEED = 0
PARAMS = {1: ["alpha"], 2: ["alpha", "beta"], 3: ["alpha", "eps0"]}
TEX = {"alpha": r"\alpha", "beta": r"\beta", "eps0": r"\hat\varepsilon_0"}
out = out_dir()


def design(form, e):
    cols = [np.ones_like(e)]
    if form == 2:
        cols += [np.log10(e), -np.log10(1 - e)]
    else:
        cols += [np.log10(e / (1 - e))]
    if form == 3:
        cols += [-1 / (e * np.log(10))]
    return np.column_stack(cols)

## Fit to the median of all 30 halos

In [ ]:
grid, raw, offsets = load_cv()
halos = list(raw)
stack = np.array([raw[h] + offsets[h] for h in halos])   # Halo 2's frame
nfw = density_fit(REF_HALO)["nfw"]
log_scale = np.log10(G**1.5 * nfw["rs"] ** 3 * nfw["rhos"] ** 0.5)

X = {k: design(k, grid) for k in PARAMS}
P = {k: np.linalg.pinv(x) for k, x in X.items()}
theta = {k: P[k] @ np.median(stack, axis=0) for k in PARAMS}
pred = {k: X[k] @ theta[k] for k in PARAMS}
mse = {(h, k): np.mean((raw[h] - pred[k] + offsets[h]) ** 2) for h in halos for k in PARAMS}


def physical(k, th):
    p = {"C": 10 ** (th[..., 0] + log_scale)}
    p.update({name: th[..., i + 1] for i, name in enumerate(PARAMS[k])})
    return p

## Halo bootstrap

Chunked so the (n_boot, n_halo, n_grid) array is never built.

In [ ]:
rng = np.random.default_rng(SEED)
boot = {k: [] for k in PARAMS}
for start in range(0, N_BOOT, 5000):
    idx = rng.integers(0, len(halos), size=(min(5000, N_BOOT - start), len(halos)))
    med = np.median(stack[idx], axis=1)
    for k in PARAMS:
        boot[k].append(med @ P[k].T)
boot = {k: np.concatenate(v) for k, v in boot.items()}

## Table 3

In [ ]:
def pm(c, draws):
    lo, hi = np.percentile(draws, [16, 84])
    return lo, hi, max(c - lo, hi - c)


def fmt(c, err):
    dec = max(0, 1 - int(np.floor(np.log10(err))))
    return f"{c:.{dec}f} \\pm {err:.{dec}f}"


rows, tex = [], []
for k in PARAMS:
    c, b = physical(k, theta[k]), physical(k, boot[k])
    row = {"form": k, "msle": np.mean([mse[h, k] for h in halos])}
    cells = []
    for name in ["C", *PARAMS[k]]:
        lo, hi, err = pm(c[name], b[name])
        row.update({name: c[name], f"{name}_p16": lo, f"{name}_p84": hi, f"{name}_pm": err})
        if name == "C":
            e = int(np.floor(np.log10(c[name])))
            cells.append(f"$C = ({fmt(c[name] / 10**e, err / 10**e)}) \\times 10^{{{e}}}$")
        else:
            cells.append(f"${TEX[name]} = {fmt(c[name], err)}$")
    rows.append(row)
    tex.append(f"{k} & {row['msle']:.4f} & " + ", ".join(cells) + r" \\")

params = pd.DataFrame(rows)
params.to_csv(out / "forms_params.csv", index=False)
pd.DataFrame([{"halo": h, **{f"msle{k}": mse[h, k] for k in PARAMS}} for h in halos]
             ).to_csv(out / "forms_per_halo.csv", index=False)
(out / "table3.tex").write_text("\n".join(tex) + "\n")
print("\n".join(tex))

## Fig. 6: best, median and worst halo by Form 1's error

In [ ]:
order = sorted(halos, key=lambda h: mse[h, 1])
picks = [("Best", order[0]), ("Median", order[len(order) // 2]), ("Worst", order[-1])]
tops, res = [], []
for panel, (name, h) in zip("abc", picks):
    model = [pred[k] - offsets[h] for k in PARAMS]
    resid = [raw[h] - m for m in model]
    tops += [raw[h], *model]
    res += resid
    np.savetxt(out / f"fig6_{panel}.dat", np.column_stack([grid, raw[h], *model, *resid]),
               header="eps raw form1 form2 form3 res1 res2 res3")
    print(f"{name}: Halo {h:02d}, MSLE " + ", ".join(f"{mse[h, k]:.4f}" for k in PARAMS))


def padded(arrs):
    lo, hi = np.min(arrs), np.max(arrs)
    return lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo)


(t0, t1), (r0, r1) = padded(tops), padded(res)
meta = [f"Y_TOP_MIN = {t0:.6g}", f"Y_TOP_MAX = {t1:.6g}", f"Y_RES_MIN = {r0:.6g}", f"Y_RES_MAX = {r1:.6g}"]
meta += [f'LABEL_{p.upper()} = "({p}) {n}: Halo {h:02d}"' for p, (n, h) in zip("abc", picks)]
(out / "fig6_meta.gnu").write_text("\n".join(meta) + "\n")

## Einasto baseline fitted to the median DF

The old Einasto baseline is Fig. 3/4's $f_{\rm Ein}$: the Eddington inversion of the Einasto fit to Halo 2's
*density*, moved to each halo with eq. (15). It was never fitted to a DF. Here Einasto is fitted to the
same target as Forms 1-3, the median of the 30 curves in Halo 2's frame, then moved to each halo with the
same offsets and scored the same way.

In $\hat\varepsilon$ the shape of an Einasto DF depends only on $\alpha$; $(r_{-2}, \rho_{-2})$ only set the
normalisation, $f \propto G^{-3/2} r_{-2}^{-3} \rho_{-2}^{-1/2}$. So the model is
$\log_{10} f = a + F_\alpha(\hat\varepsilon)$, two parameters like Form 1. $a$ is solved in closed form at
each $\alpha$; $\alpha$ is scanned on a grid and refined with a bounded 1-D minimisation. $a$ is quoted
as the shift in dex from the Einasto DF with Halo 2's $(r_{-2}, \rho_{-2})$ at the fitted $\alpha$. Error on
$\alpha$: halo bootstrap as for the forms, on the tabulated grid with parabolic refinement.

Both versions are kept: **density fit** (Halo 2's $\alpha = 0.18$, no free parameters) and **median-DF fit**.

Writes `einasto_params.csv` to `output/`; `einasto_per_halo.csv` and `einasto_fit.dat` (all baselines)
are written after the Hernquist and NFW cells below.

In [ ]:
from scipy.optimize import minimize_scalar

from Empirical_DF_Construction.core.density_fits import einasto_shape
from Empirical_DF_Construction.core.eddington import eddington_df_from_rgrid_rho

pe = density_fit(REF_HALO)["einasto"]
eddington = lambda eps_hat, r, rho, Psi=None: eddington_df_from_rgrid_rho(eps_hat, r, rho, Psi_grid=Psi)[0]


def einasto_logf(alpha, r2=pe["r2"], rho2=pe["rho2"]):
    # log10 f of an Einasto sphere on the CV grid, in Halo 2's units
    r = np.logspace(np.log10(r2 * 1e-6), np.log10(r2 * 1e4), 6000)
    return np.log10(eddington(grid, r, rho2 * einasto_shape(r, r2, alpha)))


def best_offset(target, shape):
    # least-squares a for target ~ a + shape, and the loss at that a
    a = np.mean(target - shape)
    return a, np.mean((target - shape - a) ** 2)


med = np.median(stack, axis=0)

# 1) Halo 2's density fit, no free parameters
ein_dens = einasto_logf(pe["alpha"])

# 2) fitted to the median DF: scan alpha, then refine
ALPHAS = np.round(np.arange(0.05, 0.60001, 0.0025), 4)
TAB = np.array([einasto_logf(a) for a in ALPHAS])
scan = np.array([best_offset(med, F)[1] for F in TAB])
i = int(scan.argmin())
assert 0 < i < len(ALPHAS) - 1, f"best alpha {ALPHAS[i]} is at the edge of the scan"
alpha_fit = minimize_scalar(lambda a: best_offset(med, einasto_logf(a))[1],
                            bounds=(ALPHAS[i - 1], ALPHAS[i + 1]), method="bounded",
                            options=dict(xatol=1e-6)).x
a_fit, loss_fit = best_offset(med, einasto_logf(alpha_fit))
ein_med = einasto_logf(alpha_fit) + a_fit

# halo bootstrap on the tabulated grid
N_BOOT_EIN = 20_000
rng_e = np.random.default_rng(SEED)
alpha_boot, a_ein_boot = [], []
h_step = ALPHAS[1] - ALPHAS[0]
for start in range(0, N_BOOT_EIN, 250):
    idx = rng_e.integers(0, len(halos), size=(min(250, N_BOOT_EIN - start), len(halos)))
    D = np.median(stack[idx], axis=1)[:, None, :] - TAB[None]
    L, A = D.var(axis=2), D.mean(axis=2)          # loss at best a, and that a, per alpha
    j = np.clip(L.argmin(axis=1), 1, len(ALPHAS) - 2)
    rows = np.arange(len(j))
    l0, l1, l2 = (L[rows, j + d] for d in (-1, 0, 1))
    t = 0.5 * (l0 - l2) / np.where(l0 - 2 * l1 + l2 > 0, l0 - 2 * l1 + l2, np.inf)   # parabolic step, grid units
    alpha_boot.append(ALPHAS[j] + h_step * t)
    A0, A1, A2 = (A[rows, j + d] for d in (-1, 0, 1))
    a_ein_boot.append(A1 + 0.5 * t * (A2 - A0) + 0.5 * t**2 * (A2 - 2 * A1 + A0))   # a at the refined alpha
alpha_boot, a_ein_boot = np.concatenate(alpha_boot), np.concatenate(a_ein_boot)
a16, a84 = np.percentile(alpha_boot, [16, 84])
print(f"Einasto fitted to the median DF: alpha = {alpha_fit:.4f} (+{a84 - alpha_fit:.4f} / -{alpha_fit - a16:.4f}), "
      f"a = {a_fit:+.4f} dex from Halo 2's (r_-2, rho_-2); Halo 2's density fit has alpha = {pe['alpha']}")

## Hernquist and NFW baselines

Same two versions as Einasto. Neither profile has a shape parameter, so in $\hat\varepsilon$ each DF has a
fixed shape and only the normalisation moves: $\log_{10} f = a + F(\hat\varepsilon)$.

- **density fit** (no free parameters): the DF of Halo 2's density fit. NFW is Halo 2's stored fit
  $(r_s, \rho_s)$. Hernquist, $\rho = \rho_h / [(r/a_h)(1 + r/a_h)^3]$, is not stored, so it is fitted here
  to Halo 2's density profile with the pipeline's recipe (least squares in $\ln\rho$, 2-300 kpc, bins
  with at least 200 particles, $\rho_h$ in closed form), with $a_h$ refined by a 1-D minimisation instead
  of the pipeline's 80-point grid.
- **median-DF fit** (one free parameter, $a$): the same shape shifted to the median DF. $a$ is the
  least-squares shift, quoted in dex from the density-fit normalisation, with a halo-bootstrap interval.

The Hernquist DF is analytic (Hernquist 1990). The NFW DF uses `eddington` above with the analytic
potential $\Psi = 4\pi G\rho_s r_s^2 \ln(1+x)/x$ out to $10^8 r_s$, so the infinite mass does not matter.
Writes `profiles_params.csv` to `output/`.


In [ ]:
from Empirical_DF_Construction.io.results_paths import halo_results_root


def hernquist_logf(eps_hat, a_h, M):
    # analytic isotropic Hernquist DF, f(E = eps_hat * G M / a_h)
    q = np.sqrt(np.asarray(eps_hat, float))
    F = (3 * np.arcsin(q) + q * np.sqrt(1 - q**2) * (1 - 2 * q**2) * (8 * q**4 - 8 * q**2 - 3)) / (1 - q**2) ** 2.5
    return np.log10(M / (8 * np.sqrt(2) * np.pi**3 * a_h**3 * (G * M / a_h) ** 1.5) * F)


def nfw_logf(rs, rhos):
    x = np.logspace(-6, 8, 8000)
    Psi = 4 * np.pi * G * rhos * rs**2 * np.log1p(x) / x
    return np.log10(eddington(grid, x * rs, rhos / (x * (1 + x) ** 2), Psi=Psi))


# Hernquist fit to Halo 2's density, same selection as core/density_fits.py
den = pd.read_csv(halo_results_root() / f"Halo_{REF_HALO:02d}" / "density" / "density_with_hubble_flow.csv")
fm = density_fit(REF_HALO)["fit_meta"]
sel = (den["rho_msun_kpc3"] > 0) & (den["counts"] >= fm["min_counts"]) \
      & den["r_kpc"].between(fm["r_fit_min"], fm["r_fit_max"])
r_d, lnrho_d = den.loc[sel, "r_kpc"].to_numpy(), np.log(den.loc[sel, "rho_msun_kpc3"].to_numpy())


def hern_density_loss(log_a):
    x = r_d / 10**log_a
    sh = -np.log(x * (1 + x) ** 3)
    return np.mean((lnrho_d - sh - np.mean(lnrho_d - sh)) ** 2)


lg = np.linspace(np.log10(r_d.min()), np.log10(r_d.max()), 80)
j = int(np.argmin([hern_density_loss(v) for v in lg]))
assert 0 < j < len(lg) - 1, "Hernquist scale radius at the edge of the fit range"
a_h = 10 ** minimize_scalar(hern_density_loss, bounds=(lg[j - 1], lg[j + 1]), method="bounded",
                            options=dict(xatol=1e-8)).x
x_d = r_d / a_h
rho_h = np.exp(np.mean(lnrho_d + np.log(x_d * (1 + x_d) ** 3)))
M_h = 2 * np.pi * rho_h * a_h**3
print(f"Hernquist fit to Halo 2's density: a_h = {a_h:.3f} kpc, M = {M_h:.4e} M_sun, "
      f"rms {np.sqrt(hern_density_loss(np.log10(a_h))):.3f} in ln rho over {sel.sum()} bins "
      f"(NFW {np.sqrt(nfw['sse']):.3f}, Einasto {np.sqrt(pe['sse']):.3f})")

# check the inversion on this Hernquist sphere: numerical Eddington against the analytic DF
x = np.logspace(-6, 8, 8000)
num = np.log10(eddington(grid, x * a_h, rho_h / (x * (1 + x) ** 3), Psi=G * M_h / (a_h * (1 + x))))
print(f"Eddington vs analytic Hernquist DF: max |diff| = {np.abs(num - hernquist_logf(grid, a_h, M_h)).max():.1e} dex")

shapes = {"Hernquist": hernquist_logf(grid, a_h, M_h), "NFW": nfw_logf(nfw["rs"], nfw["rhos"])}
prof_dens = dict(shapes)
prof_med, prof_a, prof_a_boot = {}, {}, {}
rng_p = np.random.default_rng(SEED)
boot_med = np.concatenate([np.median(stack[rng_p.integers(0, len(halos), size=(1000, len(halos)))], axis=1)
                           for _ in range(N_BOOT_EIN // 1000)])
for name, F in shapes.items():
    a, _ = best_offset(med, F)
    prof_med[name] = F + a
    prof_a_boot[name] = (boot_med - F).mean(axis=1)
    prof_a[name] = (a, *np.percentile(prof_a_boot[name], [16, 84]))
    print(f"{name} fitted to the median DF: a = {a:+.4f} dex "
          f"(16-84%: {prof_a[name][1]:+.4f} to {prof_a[name][2]:+.4f}) from Halo 2's density fit")


In [ ]:
# every baseline as (profile, version) -> log10 f in Halo 2's frame, in table order
base = {("Einasto", "dens"): ein_dens, ("Einasto", "med"): ein_med,
        ("NFW", "dens"): prof_dens["NFW"], ("NFW", "med"): prof_med["NFW"],
        ("Hernquist", "dens"): prof_dens["Hernquist"], ("Hernquist", "med"): prof_med["Hernquist"]}
VERSION = {"dens": "Halo 2 density fit", "med": "median-DF fit"}
N_FREE = {"dens": 0, "med": 1}
mse_base = {(h, key): np.mean((raw[h] - p + offsets[h]) ** 2) for h in halos for key, p in base.items()}
mse_ein = {(h, k): mse_base[h, ("Einasto", k)] for h in halos for k in VERSION}

models = {f"Form {k}": (pred[k], [mse[h, k] for h in halos], 1 + len(PARAMS[k])) for k in PARAMS}
for (prof, k), p in base.items():
    n = 2 if (prof, k) == ("Einasto", "med") else N_FREE[k]
    models[f"{prof}, {VERSION[k]}"] = (p, [mse_base[h, (prof, k)] for h in halos], n)
summary = pd.DataFrame([dict(model=name, n_free=n, msle_mean=np.mean(v), msle_median=np.median(v),
                             loss_vs_median_df=np.mean((p - med) ** 2))
                        for name, (p, v, n) in models.items()]).set_index("model")

pd.DataFrame([dict(model="density fit", alpha=pe["alpha"], alpha_p16=np.nan, alpha_p84=np.nan, a_dex=0.0,
                   msle=summary.loc["Einasto, Halo 2 density fit", "msle_mean"]),
              dict(model="median-DF fit", alpha=alpha_fit, alpha_p16=a16, alpha_p84=a84, a_dex=a_fit,
                   msle=summary.loc["Einasto, median-DF fit", "msle_mean"])]
             ).to_csv(out / "einasto_params.csv", index=False)
scales = {"NFW": dict(rs_kpc=nfw["rs"], rhos_msun_kpc3=nfw["rhos"]),
          "Hernquist": dict(a_h_kpc=a_h, M_msun=M_h, rho_h_msun_kpc3=rho_h)}
pd.DataFrame([dict(profile=prof, version=VERSION[k], **scales[prof],
                   a_dex=0.0 if k == "dens" else prof_a[prof][0],
                   a_p16=np.nan if k == "dens" else prof_a[prof][1],
                   a_p84=np.nan if k == "dens" else prof_a[prof][2],
                   msle=summary.loc[f"{prof}, {VERSION[k]}", "msle_mean"])
              for prof in ("NFW", "Hernquist") for k in VERSION]
             ).to_csv(out / "profiles_params.csv", index=False)
COL = {"Einasto": "ein", "NFW": "nfw", "Hernquist": "hern"}
pd.DataFrame([{"halo": h, **{f"msle{k}": mse[h, k] for k in PARAMS},
               **{f"msle_{COL[prof]}_{k}": mse_base[h, (prof, k)] for prof, k in base}} for h in halos]
             ).to_csv(out / "einasto_per_halo.csv", index=False)
p16, p84 = np.percentile(stack, [16, 84], axis=0)
np.savetxt(out / "einasto_fit.dat",
           np.column_stack([grid, med, p16, p84, pred[1], ein_med, ein_dens,
                            prof_med["NFW"], prof_dens["NFW"], prof_med["Hernquist"], prof_dens["Hernquist"]]),
           header="eps median p16 p84 form1 ein_median_fit ein_density_fit nfw_median_fit nfw_density_fit "
                  "hern_median_fit hern_density_fit   (log10 f, Halo 2's frame)")
summary.round(4)


## Table 3 rows for the profile baselines

Each baseline's own parameters at the median-DF fit. In $\hat\varepsilon$ a profile's DF fixes only
$r^3\rho^{1/2}$ (or $a^{3/2}M^{1/2}$ for Hernquist), so the radius stays at Halo 2's density-fit value and the
density normalisation absorbs the fitted shift $a$: $\rho \to \rho\,10^{-2a}$, $M \to M\,10^{-2a}$.
Errors are the halo bootstrap, quoted as in Table 3 (larger side of 16-84%). The Hernquist complexity is
its $\log_{10} f$ counted with `core.sr_complexity`, the counter that gives 11, 13 and 15 for Forms 1-3.
Writes `table3_baselines.tex` and `baselines_params.csv` to `output/`.


In [ ]:
import sympy as sp
from Empirical_DF_Construction.core.sr_complexity import complexity as sr_complexity

xs = sp.Symbol("x")
qs = sp.sqrt(xs)
hern_expr = sp.Float(-1.2) + sp.log(3 * sp.asin(qs) + qs * sp.sqrt(1 - xs) * (1 - 2 * xs) * (8 * xs**2 - 8 * xs - 3)) \
            - sp.Rational(5, 2) * sp.log(1 - xs)
cplx_hern = sr_complexity(hern_expr)
assert [sr_complexity(sp.sympify(e)) for e in ("-2.7 + 2.3*log(-x/(x-1))", "-2.6 + 2.5*log(x) - 2.2*log(1-x)",
                                                "-2.6 + 2.3*log(-x/(x-1)) - 0.036/x")] == [11, 13, 15]


def sci(c, err, unit):
    e = int(np.floor(np.log10(c)))
    return f"({fmt(c / 10**e, err / 10**e)})\\times 10^{{{e}}}~\\mathrm{{{unit}}}"


RHO_U, M_U = r"M_\odot\,kpc^{-3}", r"M_\odot"
a_draws = {"Einasto": (a_fit, a_ein_boot), "NFW": (prof_a["NFW"][0], prof_a_boot["NFW"]),
           "Hernquist": (prof_a["Hernquist"][0], prof_a_boot["Hernquist"])}
norm = {"Einasto": pe["rho2"], "NFW": nfw["rhos"], "Hernquist": M_h}
spec = {  # radius line, normalisation symbol, unit, expression, complexity
    "Einasto": (rf"r_{{-2}} &= {pe['r2']:.1f}~\mathrm{{kpc}}~\text{{(fixed)}}", r"\rho_{-2}", RHO_U, "N/A", "N/A"),
    "NFW": (rf"r_s &= {nfw['rs']:.1f}~\mathrm{{kpc}}~\text{{(fixed)}}", r"\rho_s", RHO_U, "N/A", "N/A"),
    "Hernquist": (rf"a &= {a_h:.1f}~\mathrm{{kpc}}~\text{{(fixed)}}", "M", M_U, r"\eqref{eq:hernquist_df}", str(cplx_hern)),
}
rows_b, tex_b = [], []
for prof, (a_c, a_b) in a_draws.items():
    rad, sym, unit, expr, cx = spec[prof]
    n_c, n_b = norm[prof] * 10 ** (-2 * a_c), norm[prof] * 10 ** (-2 * a_b)
    lo, hi, err = pm(n_c, n_b)
    row = dict(profile=prof, msle=summary.loc[f"{prof}, median-DF fit", "msle_mean"], norm=n_c,
               norm_p16=lo, norm_p84=hi, norm_pm=err, a_dex=a_c)
    lines = []
    if prof == "Einasto":
        alo, ahi, aerr = pm(alpha_fit, alpha_boot)
        row.update(alpha=alpha_fit, alpha_p16=alo, alpha_p84=ahi, alpha_pm=aerr)
        lines.append(rf"\alpha_E &= {fmt(alpha_fit, aerr)}")
    lines += [rad, rf"{sym} &= {sci(n_c, err, unit)}"]
    rows_b.append(row)
    tex_b.append(f"{prof} & {expr} & {cx} & {row['msle']:.4f}\n& $\\displaystyle \\begin{{aligned}}\n"
                 + "\\\\\n".join(lines) + "\n\\end{aligned}$ \\\\\n\\hline")

pd.DataFrame(rows_b).to_csv(out / "baselines_params.csv", index=False)
(out / "table3_baselines.tex").write_text("\n".join(tex_b) + "\n")
print("\n".join(tex_b))


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"mathtext.fontset": "stix", "font.family": "STIXGeneral"})
COLOR = {"Form 1": "C0", "Einasto": "C1", "NFW": "C2", "Hernquist": "C3"}
LS = {"med": "-", "dens": "--"}   # median-DF fit solid, Halo 2 density fit dashed, same colour
curves = {"Form 1": (pred[1], dict(color=COLOR["Form 1"], ls="-", lw=2))}
curves.update({f"{prof}, {VERSION[k]}": (p, dict(color=COLOR[prof], ls=LS[k], lw=1.3)) for (prof, k), p in base.items()})

fig, ax = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw=dict(wspace=0.28))
ax[0].fill_between(grid, p16, p84, color="0.85", lw=0, label="30 halos, 16-84%")
ax[0].plot(grid, med, color="k", lw=1.5, label="median")
ax[1].fill_between(grid, p16 - med, p84 - med, color="0.85", lw=0)
ax[1].axhline(0, color="k", lw=0.8)
for name, (y, kw) in curves.items():
    ax[0].plot(grid, y, label=f"{name} (MSLE {summary.loc[name, 'msle_mean']:.4f})", **kw)
    ax[1].plot(grid, y - med, **kw)
ax[0].set_ylabel(r"$\log_{10} f$ (Halo 2's frame)")
ax[0].legend(frameon=False, fontsize=7.5, loc="upper left")
ax[1].set_ylabel(r"model $-$ median [dex]")
ax[1].set_ylim(-1.0, 1.0)
for a, t in zip(ax[:2], ["(a) Median DF and models", "(b) Residual from the median"]):
    a.set_xlabel(r"$\hat\varepsilon$")
    a.set_title(t, loc="left")
    a.grid(True, alpha=0.3)

# (c) per-halo MSLE of each median-DF fit against Form 1
f1 = np.array([mse[h, 1] for h in halos])
ys = {prof: np.array([mse_base[h, (prof, "med")] for h in halos]) for prof in ("Einasto", "NFW", "Hernquist")}
for prof, y in ys.items():
    ax[2].scatter(f1, y, s=30, c=COLOR[prof], edgecolors="k", linewidths=0.4, label=f"{prof}, median-DF fit")
allv = np.concatenate([f1, *ys.values()])
lim = [allv.min() * 0.7, allv.max() * 1.4]
ax[2].plot(lim, lim, color="0.5", lw=0.8, ls=":")
ax[2].set(xscale="log", yscale="log", xlim=lim, ylim=lim,
          xlabel="Halo MSLE, Form 1", ylabel="Halo MSLE, profile baseline")
ax[2].set_title("(c) Per halo (above the line: baseline worse)", loc="left")
ax[2].legend(frameon=False, fontsize=8, loc="lower right")
ax[2].grid(True, alpha=0.3, which="both")
plt.show()
